# SentryNet -- Feature Engineering (including graph features)

In [ ]:
from sentrynet.config import DATA_DIR

TRANSACTION_PATH = DATA_DIR / "train_transaction.csv"
IDENTITY_PATH = DATA_DIR / "train_identity.csv"
DATA_AVAILABLE = TRANSACTION_PATH.exists()

if not DATA_AVAILABLE:
    print(f"Dataset not found at {TRANSACTION_PATH}. Download the IEEE-CIS "
          "Fraud Detection dataset from Kaggle and place it under data/ to run this notebook.")

## Transaction-level features (label-independent, safe to compute before the split)

In [ ]:
if DATA_AVAILABLE:
    from sentrynet.features.velocity import transaction_velocity
    from sentrynet.features.recency import time_since_last
    from sentrynet.features.geo import addr_change_flag
    from sentrynet.graph.fingerprint import build_card_entity_ids

    df["card_entity_id"] = build_card_entity_ids(df)
    df["velocity_1h"] = transaction_velocity(df, "card_entity_id", "TransactionDT", 3600)
    df["time_since_last"] = time_since_last(df, "card_entity_id", "TransactionDT")
    df["addr_changed"] = addr_change_flag(df, "card_entity_id", "TransactionDT")

    # Merchant risk encoding is intentionally NOT computed here: it is fit on
    # isFraud, so it must be fit after the temporal split (train only) to avoid
    # leakage -- see the Modeling and Evaluation notebook.

## Graph features: device fingerprint + reconstructed card identity

In [ ]:
# NOTE: the reconstructed card identity (card_entity_id above) is a
# probabilistic heuristic, not a verified cardholder ID -- see
# sentrynet.graph.fingerprint.build_card_entity_ids docstring.
if DATA_AVAILABLE:
    from sentrynet.graph.fingerprint import build_device_fingerprints
    from sentrynet.graph.build import build_bipartite_graph
    from sentrynet.graph.features import extract_entity_features
    from sentrynet.graph.join import transaction_entity_features

    df["device_fingerprint"] = build_device_fingerprints(df)
    g = build_bipartite_graph(
        df, transaction_id_col="TransactionID",
        entity_cols=("device_fingerprint", "card_entity_id"),
    )
    entity_features = extract_entity_features(g)
    graph_features = transaction_entity_features(
        df, entity_cols=("device_fingerprint", "card_entity_id"), entity_features=entity_features
    )
    df = pd.concat([df, graph_features], axis=1)